# 🏎️ F1 Predictions: 2026 Japanese Grand Prix (Round 3)
## Suzuka Circuit — March 27-29, 2026

**Model version:** v0.2 (2026-weighted, constructor-aware)  
**Data sources:** FastF1 API (2023-2026)  
**Last updated:** March 24, 2026

### Methodology (v0.2)
The 2026 season introduced **massive regulation changes** — new power units (50/50 electric/ICE), 
active aerodynamics, smaller/lighter cars, and 100% sustainable fuels. This means:

- **Pre-2026 data has limited predictive value** for car performance — the pecking order has been reshuffled
- **2026 race results are the primary signal** (only 2 races, but they're the only relevant data for constructor strength)
- **Historical data is still useful for:** driver skill (racecraft, consistency), teammate comparisons, and track knowledge

**Our model combines:**
1. **Constructor Strength (50%)** — 2026-only team performance tier
2. **Driver Skill (20%)** — teammate delta + historical racecraft  
3. **2026 Form (20%)** — actual finishing positions this season
4. **Suzuka Track Factor (10%)** — limited weight since cars are new, but driver track knowledge still matters


In [ ]:
import fastf1
import pandas as pd
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

fastf1.Cache.enable_cache('../data/cache')
plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 6)

print("Setup complete ✓")


## 1. Data Collection

Pulling race results from:
- **2026 Rounds 1-2** (Australia, China) — PRIMARY data for constructor strength + 2026 form
- **2023-2025 Japanese GP** — Suzuka-specific driver knowledge (discounted for car changes)
- **Last 5 races of 2025** — driver skill baseline (heavily discounted)


In [ ]:
def get_race_results(year, round_num):
    session = fastf1.get_session(year, round_num, 'R')
    session.load(telemetry=False, weather=False, messages=False)
    df = session.results[['DriverNumber', 'FullName', 'TeamName', 'Position', 'GridPosition', 'Points', 'Status']].copy()
    df['Year'] = year
    df['Round'] = round_num
    df['RaceName'] = session.event['EventName']
    df['Position'] = pd.to_numeric(df['Position'], errors='coerce')
    df['GridPosition'] = pd.to_numeric(df['GridPosition'], errors='coerce')
    return df

all_results = []

# 2026 races (PRIMARY)
for rnd in [1, 2]:
    all_results.append(get_race_results(2026, rnd))

# Japanese GPs 2023-2025 (for Suzuka track knowledge)
for year in [2023, 2024, 2025]:
    schedule = fastf1.get_event_schedule(year)
    japan = schedule[schedule['EventName'].str.contains('Japan')]
    if len(japan) > 0:
        all_results.append(get_race_results(year, int(japan.iloc[0]['RoundNumber'])))

# Last 5 races of 2025 (for driver skill baseline)
schedule_2025 = fastf1.get_event_schedule(2025)
races_2025 = schedule_2025[schedule_2025['RoundNumber'] > 0]
for _, race in races_2025.tail(5).iterrows():
    all_results.append(get_race_results(2025, int(race['RoundNumber'])))

df = pd.concat(all_results, ignore_index=True)
print(f"Collected {len(df)} driver-race entries across {df[['Year','Round']].drop_duplicates().shape[0]} races")
print(f"\nRaces by era:")
for _, row in df[['Year','RaceName']].drop_duplicates().sort_values('Year').iterrows():
    era = "🟢 2026 (primary)" if row['Year'] == 2026 else "🟡 historical (discounted)"
    print(f"  {row['Year']} {row['RaceName']:30s} {era}")


## 2. Constructor Strength (2026 Only)

This is the **biggest predictor** in a regulation-change year. Which teams nailed the new rules?
We calculate constructor strength purely from the 2 races of 2026 data.


In [ ]:
# Constructor strength from 2026 results ONLY
data_2026 = df[df['Year'] == 2026].copy()

constructor_stats = data_2026.groupby('TeamName').agg(
    avg_position=('Position', 'mean'),
    best_finish=('Position', 'min'),
    avg_grid=('GridPosition', 'mean'),
    races=('Position', 'count')
).reset_index().sort_values('avg_position')

# Assign tiers
def assign_tier(avg_pos):
    if avg_pos <= 3: return 'Tier 1 (Front-runners)'
    elif avg_pos <= 7: return 'Tier 2 (Strong midfield)'
    elif avg_pos <= 12: return 'Tier 3 (Midfield)'
    elif avg_pos <= 16: return 'Tier 4 (Lower midfield)'
    else: return 'Tier 5 (Backmarkers)'

constructor_stats['Tier'] = constructor_stats['avg_position'].apply(assign_tier)

# Constructor strength score (lower = better, 1-22 scale)
constructor_strength = dict(zip(constructor_stats['TeamName'], constructor_stats['avg_position']))

print("=" * 70)
print("  🏗️  CONSTRUCTOR STRENGTH — Based on 2026 Results Only")
print("=" * 70)
print(f"\n  {'Team':22s}  {'Avg Finish':>10s}  {'Best':>5s}  {'Avg Grid':>9s}  {'Tier'}")
print("  " + "-" * 65)
for _, row in constructor_stats.iterrows():
    print(f"  {row['TeamName']:22s}  P{row['avg_position']:>8.1f}  P{row['best_finish']:>3.0f}  P{row['avg_grid']:>7.1f}  {row['Tier']}")

# Visualize
fig, ax = plt.subplots(figsize=(12, 5))
teams = constructor_stats.sort_values('avg_position')
colors_map = {
    'Tier 1 (Front-runners)': '#ff006e',
    'Tier 2 (Strong midfield)': '#8338ec', 
    'Tier 3 (Midfield)': '#3a86ff',
    'Tier 4 (Lower midfield)': '#06ffa5',
    'Tier 5 (Backmarkers)': '#666666'
}
bar_colors = [colors_map.get(t, '#999') for t in teams['Tier']]
bars = ax.barh(teams['TeamName'], teams['avg_position'], color=bar_colors)
ax.set_xlabel('Average Finish Position (lower = better)', fontsize=12)
ax.set_title('2026 Constructor Strength (Australia + China)', fontsize=14, fontweight='bold')
ax.invert_xaxis()
for bar, val in zip(bars, teams['avg_position']):
    ax.text(val - 0.3, bar.get_y() + bar.get_height()/2, f'P{val:.1f}', va='center', ha='right', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/constructor_strength.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Driver Skill Rating

Measures **driver quality independent of car performance** using:
- **Teammate delta:** How much does each driver beat/lose to their teammate? This transfers across regulation changes.
- **Positions gained in races:** Some drivers consistently move forward from their grid position (racecraft).
- **Consistency:** Low variance in finishing position = reliable.


In [ ]:
# Teammate comparison (2026 data — most relevant)
teammate_deltas = {}
teams_2026 = data_2026.groupby(['Year', 'Round', 'TeamName'])

for (year, rnd, team), group in teams_2026:
    if len(group) >= 2:
        drivers = group.sort_values('Position')
        d1 = drivers.iloc[0]
        d2 = drivers.iloc[1]
        # The driver who finished ahead gets a positive delta
        delta = d2['Position'] - d1['Position']
        
        if d1['FullName'] not in teammate_deltas:
            teammate_deltas[d1['FullName']] = []
        if d2['FullName'] not in teammate_deltas:
            teammate_deltas[d2['FullName']] = []
        
        teammate_deltas[d1['FullName']].append(delta)
        teammate_deltas[d2['FullName']].append(-delta)

# Positions gained (race vs grid) — all data, weighted toward 2026
positions_gained = {}
for _, row in df.iterrows():
    if pd.notna(row['Position']) and pd.notna(row['GridPosition']):
        driver = row['FullName']
        gain = row['GridPosition'] - row['Position']
        weight = 3.0 if row['Year'] == 2026 else 0.5  # 2026 weighted 6x more
        if driver not in positions_gained:
            positions_gained[driver] = []
        positions_gained[driver].extend([gain] * int(weight * 2))

# Build driver skill scores
grid_2026 = data_2026['FullName'].unique()
driver_skills = []

for driver in grid_2026:
    tm_delta = np.mean(teammate_deltas.get(driver, [0]))
    avg_gain = np.mean(positions_gained.get(driver, [0]))
    
    # Skill score: positive = better than average
    skill_score = tm_delta * 0.6 + avg_gain * 0.4
    
    team = data_2026[data_2026['FullName'] == driver]['TeamName'].iloc[0]
    teammate = data_2026[(data_2026['TeamName'] == team) & (data_2026['FullName'] != driver)]['FullName'].unique()
    tm_name = teammate[0] if len(teammate) > 0 else 'N/A'
    
    driver_skills.append({
        'Driver': driver,
        'Team': team,
        'Teammate': tm_name,
        'Teammate_Delta': tm_delta,
        'Avg_Positions_Gained': avg_gain,
        'Skill_Score': skill_score
    })

skill_df = pd.DataFrame(driver_skills).sort_values('Skill_Score', ascending=False)

print("=" * 75)
print("  👤 DRIVER SKILL RATINGS")
print("=" * 75)
print(f"\n  {'Driver':22s}  {'Team':18s}  {'vs Teammate':>12s}  {'Pos Gained':>10s}  {'Skill':>6s}")
print("  " + "-" * 72)
for _, row in skill_df.iterrows():
    tm_str = f"+{row['Teammate_Delta']:.1f}" if row['Teammate_Delta'] > 0 else f"{row['Teammate_Delta']:.1f}"
    gain_str = f"+{row['Avg_Positions_Gained']:.1f}" if row['Avg_Positions_Gained'] > 0 else f"{row['Avg_Positions_Gained']:.1f}"
    print(f"  {row['Driver']:22s}  {row['Team']:18s}  {tm_str:>12s}  {gain_str:>10s}  {row['Skill_Score']:>+5.1f}")


## 4. Suzuka Track Factor (Discounted)

Historical Suzuka performance — **heavily discounted** because the cars are completely different. 
But driver track knowledge (knowing the corners, braking points, racing lines) still has some value.
Weight: only 10% of the composite score.


In [ ]:
# Suzuka results 2023-2025
suzuka_data = df[(df['RaceName'].str.contains('Japan', case=False)) & (df['Year'] < 2026)].copy()
suzuka_avg = suzuka_data.groupby('FullName').agg(
    avg_pos=('Position', 'mean'),
    avg_grid=('GridPosition', 'mean'),
    races=('Position', 'count')
).reset_index()
suzuka_avg['track_advantage'] = suzuka_avg['avg_grid'] - suzuka_avg['avg_pos']

# Map to 2026 grid
suzuka_factors = dict(zip(suzuka_avg['FullName'], suzuka_avg['track_advantage']))

print("=== Suzuka Track Knowledge (2023-2025, discounted for new regs) ===\n")
print(f"  {'Driver':22s}  {'Avg Finish':>10s}  {'Positions Gained':>16s}  {'Races':>5s}")
print("  " + "-" * 58)
for _, row in suzuka_avg.sort_values('avg_pos').iterrows():
    if row['FullName'] in grid_2026:
        gain = f"+{row['track_advantage']:.1f}" if row['track_advantage'] > 0 else f"{row['track_advantage']:.1f}"
        print(f"  {row['FullName']:22s}  P{row['avg_pos']:>8.1f}  {gain:>16s}  {int(row['races']):>5d}")

# Drivers without Suzuka F1 history
no_suzuka = [d for d in grid_2026 if d not in suzuka_factors]
if no_suzuka:
    print(f"\n  No Suzuka history: {', '.join(no_suzuka)}")


## 5. Composite Prediction — Japan GP

**Weighting:**
- Constructor Strength: **50%** (which car are you in?)
- Driver Skill: **20%** (how good are you relative to your teammate?)  
- 2026 Form: **20%** (where have you been finishing?)
- Suzuka Factor: **10%** (do you know this track?)


In [ ]:
# Build composite predictions
predictions = []

for driver in grid_2026:
    team = data_2026[data_2026['FullName'] == driver]['TeamName'].iloc[0]
    
    # 1. Constructor strength (avg team finish position in 2026)
    constructor_score = constructor_strength.get(team, 15)
    
    # 2. Driver skill (teammate delta + positions gained)
    skill = skill_df[skill_df['Driver'] == driver]['Skill_Score'].values[0]
    # Convert to position adjustment: positive skill = better position
    skill_adjustment = -skill * 1.5  # Scale: +1 skill ≈ 1.5 positions better
    
    # 3. 2026 form (average finish this season)
    form_2026 = data_2026[data_2026['FullName'] == driver]['Position'].mean()
    
    # 4. Suzuka factor
    suzuka = suzuka_factors.get(driver, 0)
    suzuka_adjustment = -suzuka * 0.3  # Discounted — cars are different
    
    # Composite: weighted blend
    composite = (
        0.50 * constructor_score +
        0.20 * (constructor_score + skill_adjustment) +
        0.20 * form_2026 +
        0.10 * (form_2026 + suzuka_adjustment)
    )
    
    predictions.append({
        'Driver': driver,
        'Team': team,
        'Constructor': constructor_score,
        'Skill': skill,
        'Form_2026': form_2026,
        'Suzuka_Adj': suzuka,
        'Composite': composite
    })

pred_df = pd.DataFrame(predictions).sort_values('Composite').reset_index(drop=True)
pred_df['Predicted_Position'] = range(1, len(pred_df) + 1)

print("=" * 80)
print("  🏁 PREDICTED FINISH ORDER — 2026 JAPANESE GRAND PRIX")
print("=" * 80)
print(f"\n  {'Pos':>3s}  {'Driver':22s}  {'Team':18s}  {'Car':>4s}  {'Skill':>6s}  {'Form':>5s}  {'Suzuka':>7s}")
print("  " + "-" * 75)
for _, row in pred_df.iterrows():
    suzuka_str = f"+{row['Suzuka_Adj']:.1f}" if row['Suzuka_Adj'] > 0 else f"{row['Suzuka_Adj']:.1f}" if row['Suzuka_Adj'] != 0 else "  new"
    print(f"  P{row['Predicted_Position']:>2d}  {row['Driver']:22s}  {row['Team']:18s}  P{row['Constructor']:>3.1f}  {row['Skill']:>+5.1f}  P{row['Form_2026']:>4.1f}  {suzuka_str:>7s}")


## 6. Podium Probabilities (Monte Carlo)

10,000 race simulations with randomness calibrated to actual F1 variance.
With only 2 races of data, we use **wider uncertainty bands** than we would mid-season.


In [ ]:
np.random.seed(42)
n_sims = 10000

# Wider noise for early season (less data = less confidence)
noise_std = 4.0  # ~4 position standard deviation

podium_counts = defaultdict(int)
win_counts = defaultdict(int)
top5_counts = defaultdict(int)
top10_counts = defaultdict(int)
position_totals = defaultdict(list)

for _ in range(n_sims):
    sim_results = []
    for _, row in pred_df.iterrows():
        noise = np.random.normal(0, noise_std)
        sim_results.append((row['Driver'], row['Composite'] + noise))
    
    sim_results.sort(key=lambda x: x[1])
    
    for pos, (driver, _) in enumerate(sim_results):
        position_totals[driver].append(pos + 1)
        if pos == 0: win_counts[driver] += 1
        if pos < 3: podium_counts[driver] += 1
        if pos < 5: top5_counts[driver] += 1
        if pos < 10: top10_counts[driver] += 1

# Build probability table
prob_data = []
for _, row in pred_df.iterrows():
    d = row['Driver']
    prob_data.append({
        'Driver': d,
        'Team': row['Team'],
        'Pred': row['Predicted_Position'],
        'Win%': win_counts[d] / n_sims * 100,
        'Podium%': podium_counts[d] / n_sims * 100,
        'Top5%': top5_counts[d] / n_sims * 100,
        'Top10%': top10_counts[d] / n_sims * 100,
        'Avg_Sim_Pos': np.mean(position_totals[d]),
        'Pos_StdDev': np.std(position_totals[d])
    })

prob_df = pd.DataFrame(prob_data).sort_values('Win%', ascending=False)

print("=" * 80)
print("  🎲 RACE OUTCOME PROBABILITIES (10,000 simulations)")
print("=" * 80)
print(f"\n  {'Driver':22s}  {'Team':18s}  {'Win%':>6s}  {'Podium':>7s}  {'Top 5':>6s}  {'Top 10':>7s}")
print("  " + "-" * 72)
for _, row in prob_df.head(15).iterrows():
    win_bar = '█' * int(row['Win%'] / 2)
    print(f"  {row['Driver']:22s}  {row['Team']:18s}  {row['Win%']:>5.1f}%  {row['Podium%']:>6.1f}%  {row['Top5%']:>5.1f}%  {row['Top10%']:>6.1f}%  {win_bar}")

# Podium probability chart
fig, ax = plt.subplots(figsize=(12, 7))
top12 = prob_df.head(12).sort_values('Podium%')
colors = ['#ff006e' if p > 40 else '#8338ec' if p > 15 else '#3a86ff' if p > 5 else '#06ffa5' for p in top12['Podium%']]
bars = ax.barh(top12['Driver'] + '  (' + top12['Team'] + ')', top12['Podium%'], color=colors)
ax.set_xlabel('Podium Probability (%)', fontsize=12)
ax.set_title('2026 Japanese GP — Podium Probabilities (10K simulations)', fontsize=14, fontweight='bold')
for bar, val in zip(bars, top12['Podium%']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('../data/podium_probs_v2.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. F1 Fantasy Recommendations

### Scoring System (2026)
**Qualifying:** P1=10, P2=9, ... P10=1, NC=-5  
**Race:** P1=25, P2=18, P3=15, P4=12, P5=10, P6=8, P7=6, P8=4, P9=2, P10=1, DNF=-20  
**Bonuses:** +1 per position gained, +1 per overtake, +10 fastest lap, +10 Driver of the Day  
**Constructor bonuses:** Both Q3=+10, both Q2=+5, pit stops under 2.2s=+5 to +10  
**Budget:** $100M for 5 drivers + 2 constructors | 1 driver gets 2x boost  

### Strategy
The model targets **expected fantasy points** not just finish position. A driver who starts P15 and finishes P8 
may score MORE fantasy points than a driver who starts P3 and finishes P4 (due to position-gained bonuses).


In [ ]:
# F1 Fantasy expected points calculation
race_points = {1: 25, 2: 18, 3: 15, 4: 12, 5: 10, 6: 8, 7: 6, 8: 4, 9: 2, 10: 1}
quali_points = {1: 10, 2: 9, 3: 8, 4: 7, 5: 6, 6: 5, 7: 4, 8: 3, 9: 2, 10: 1}

fantasy = []
for _, row in pred_df.iterrows():
    driver = row['Driver']
    pred_pos = row['Predicted_Position']
    form = row['Form_2026']
    
    # Expected qualifying position (approximate from form + constructor)
    est_quali = max(1, min(22, int(row['Composite'] * 0.9)))  # Quali tends to follow car pace
    
    # Expected race points (from simulation distribution)
    exp_race_pts = 0
    exp_quali_pts = 0
    exp_positions_gained = 0
    exp_overtakes = 0
    
    sim_positions = position_totals[driver]
    for sim_pos in sim_positions:
        exp_race_pts += race_points.get(sim_pos, 0)
    exp_race_pts /= n_sims
    
    # Quali points estimate
    exp_quali_pts = quali_points.get(est_quali, 0)
    
    # Positions gained estimate (grid → finish)
    avg_sim_pos = np.mean(sim_positions)
    exp_positions_gained = max(0, est_quali - avg_sim_pos)
    
    # Overtake estimate (~0.5 per position gained for midfield, less for front)
    exp_overtakes = exp_positions_gained * 0.5
    
    # Total expected fantasy points
    total_fantasy = exp_race_pts + exp_quali_pts + exp_positions_gained + exp_overtakes
    
    # Value score (points per estimated price — higher = better value)
    # Rough price estimates based on 2026 constructor tier
    price_estimate = {
        'Mercedes': 28, 'Ferrari': 25, 'McLaren': 22, 'Red Bull Racing': 20,
        'Aston Martin': 12, 'Alpine': 10, 'Racing Bulls': 9, 'Haas F1 Team': 8,
        'Williams': 8, 'Audi': 7, 'Cadillac': 6
    }
    est_price = price_estimate.get(row['Team'], 10)
    value = total_fantasy / est_price if est_price > 0 else 0
    
    fantasy.append({
        'Driver': driver,
        'Team': row['Team'],
        'Pred_Finish': pred_pos,
        'Est_Quali': est_quali,
        'Race_Pts': exp_race_pts,
        'Quali_Pts': exp_quali_pts,
        'Bonus_Pts': exp_positions_gained + exp_overtakes,
        'Total_Fantasy': total_fantasy,
        'Est_Price': est_price,
        'Value': value
    })

fantasy_df = pd.DataFrame(fantasy).sort_values('Total_Fantasy', ascending=False)

print("=" * 85)
print("  🏆 F1 FANTASY RECOMMENDATIONS — Japan GP")
print("=" * 85)
print(f"\n  {'Driver':22s}  {'Team':16s}  {'Pred':>4s}  {'Race':>5s}  {'Qual':>5s}  {'Bonus':>5s}  {'TOTAL':>6s}  {'~$M':>4s}  {'Value':>5s}")
print("  " + "-" * 82)
for _, row in fantasy_df.iterrows():
    star = '⭐' if row['Value'] > 2.0 else '  '
    print(f"  {row['Driver']:22s}  {row['Team']:16s}  P{row['Pred_Finish']:>2.0f}  {row['Race_Pts']:>5.1f}  {row['Quali_Pts']:>5.0f}  {row['Bonus_Pts']:>5.1f}  {row['Total_Fantasy']:>5.1f}  ${row['Est_Price']:>3.0f}  {row['Value']:>4.2f}  {star}")

# Best value picks
print("\n  " + "=" * 55)
print("  💡 RECOMMENDED LINEUP")
print("  " + "=" * 55)

# Sort by value for budget optimization
value_picks = fantasy_df.sort_values('Value', ascending=False)
print("\n  🔥 Best value picks (points per $M):")
for _, row in value_picks.head(5).iterrows():
    print(f"     → {row['Driver']:22s} ({row['Team']:16s}) — {row['Total_Fantasy']:.1f} pts / ${row['Est_Price']}M = {row['Value']:.2f} val")

print("\n  🚀 Best 2x Boost candidate:")
boost = fantasy_df.iloc[0]
print(f"     → {boost['Driver']} ({boost['Team']}) — {boost['Total_Fantasy']:.1f} expected pts × 2 = {boost['Total_Fantasy']*2:.1f}")

print("\n  🏗️ Constructor picks:")
# Constructor scoring = sum of both drivers + quali bonus + pit bonus
constructor_fantasy = fantasy_df.groupby('Team').agg(
    combined_pts=('Total_Fantasy', 'sum'),
    est_price=('Est_Price', 'first')
).sort_values('combined_pts', ascending=False)
for team, row in constructor_fantasy.head(3).iterrows():
    print(f"     → {team:22s} — {row['combined_pts']:.1f} combined driver pts")


## 8. Head-to-Head Matchups

Key teammate battles and driver-vs-driver predictions for betting and fantasy.


In [ ]:
# Teammate head-to-head probabilities
teams = data_2026.groupby('TeamName')['FullName'].unique()

print("=" * 60)
print("  🤝 TEAMMATE HEAD-TO-HEAD PROBABILITIES")
print("=" * 60)
print()

for team, drivers in teams.items():
    if len(drivers) >= 2:
        d1, d2 = drivers[0], drivers[1]
        
        # Count sim wins
        d1_wins = 0
        for i in range(n_sims):
            if position_totals[d1][i] < position_totals[d2][i]:
                d1_wins += 1
        
        d1_pct = d1_wins / n_sims * 100
        d2_pct = 100 - d1_pct
        
        # Bar visualization
        bar_len = 40
        d1_bars = int(d1_pct / 100 * bar_len)
        d2_bars = bar_len - d1_bars
        
        winner = d1 if d1_pct > 50 else d2
        
        print(f"  {team}")
        print(f"  {d1:20s} {'█' * d1_bars}{'░' * d2_bars} {d2:>20s}")
        print(f"  {d1_pct:>19.1f}% {'':40s} {d2_pct:<.1f}%")
        print()


## 9. Summary & Confidence Notes

### Model Confidence: LOW-MEDIUM
- Only 2 races of 2026 data — predictions will improve significantly after Japan (3 data points)
- Constructor strength tiers are clear but exact positions within tiers have high uncertainty
- The ±4 position standard deviation in simulations reflects this uncertainty

### Key Predictions
- **Mercedes 1-2 most likely** — they've been dominant in both 2026 races
- **Ferrari solidly in P3-P4 territory** — Hamilton and Leclerc are the clear second-best team
- **Midfield is wide open** — Haas, Alpine, Racing Bulls, and Red Bull are all fighting for P5-P12
- **McLaren struggling** — a shocking fall from 2025 dominance, possibly the biggest loser of the regulation changes

### What Could Change This
- **Rain** — Suzuka weather is unpredictable in March. Rain scrambles the grid.
- **Red Bull upgrades** — if they've found something in the 2-week break, Verstappen could surge
- **Safety cars** — Suzuka's narrow sections produce incidents. These favor midfield teams.
- **Practice pace** — FP1/FP2/FP3 data will be crucial for refining predictions

### Next Steps
- [ ] Update after FP1/FP2 (Friday)
- [ ] Re-run after qualifying (Saturday) with actual grid positions
- [ ] Post-race analysis: how did the model do?
- [ ] Improve constructor strength model with 3rd data point
